# Chapter 2 — Fact Overlay, Why-not, and Frontier

Three diagnostic capabilities that build on the SDK + Check foundation from chapter 1:

1. **Q3 Fact Overlay** (`check_fact_overlay_binding`) — ask a what-if question without writing the ledger.
2. **Q4 Why-not Universe** (`check_why_not_universe`) — partition a finite candidate universe into red/green rows and locate per-row failures.
3. **Q5 Frontier Trace** (`evaluate_native_where_frontier`) — drop into the evaluator layer and see where the where-body collapses on the failing path.

Each phase imports its scenario from `round_story_full_demo.py` to stay aligned with the smoke harness.

**Series navigation**

- Previous: `01_sdk_check_diagnose.ipynb` (SDK fixture, Q1 Check, Q2 Diagnose).
- Next: `03_proofframe_rule_overlays.ipynb` (Batch 4 ProofFrame Rechecker + Batch 5 rule overlays).
- Integrated walkthrough: `round_story_full_demo.py`.

## Setup

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'examples').exists() and (_repo_root.parent / 'examples').exists():
    _repo_root = _repo_root.parent
for sub in ('src', 'examples'):
    candidate = _repo_root / sub
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import round_story_full_demo as demo  # noqa: E402

fixture = demo._build_fixture()
print(f'Fixture ready: {len(fixture.people)} people seeded.')

## 1. Q3 Fact Overlay — what if Alice were 30?

`check_fact_overlay_binding(...)` evaluates a binding twice — `before` (raw ledger) and `after` (ledger view with one or more `FactValueOverride` actions applied) — and returns a structured diff. The ledger is never mutated.

The phase below overlays Alice's age from `25` to `30` and checks whether the eligibility rule (which here matches age=30 region=us) flips from `failed` to `passed`.

In [ ]:
fact_overlay_request, fact_overlay_result, _overlay = demo._phase_fact_overlay(
    fixture, verbose=True
)

print(f'\noverall status   = {fact_overlay_result.status}')
print(f'before status    = {fact_overlay_result.before.status}')
print(f'after status     = {fact_overlay_result.after.status}')
print(f'status_changed   = {fact_overlay_result.diff.status_changed}')
print(f'matched_count_delta = {fact_overlay_result.diff.matched_count_delta}')

**Ledger immutability:** `_phase_fact_overlay` asserts that the ledger digest is byte-equal before and after the overlay call. Fact overlays are pure projection-merge operations; no `set/add/retract` is performed.

## 2. Q4 Why-not Universe — who passes and who fails, and why?

Given an explicit finite candidate universe, `check_why_not_universe(...)` returns:

- `green` — bindings that match the derivation
- `red` — failing rows, each carrying a `WhyNotRowDiagnostic` with status, failure_kind, granularity, and an `atom_locator` for the responsible body atom

The phase below asks the universe `{alice@30/us, bob@30/eu, carol@30/us}` against the same plan.

In [ ]:
why_not_request, why_not_result = demo._phase_why_not(fixture, verbose=True)

print(f'\nstatus       = {why_not_result.status}')
print(f'green count  = {len(why_not_result.green)}')
print(f'red count    = {len(why_not_result.red)}')
for row in why_not_result.red:
    locator = row.diagnostic.atom_locator
    print(
        f'  red {row.binding}: failed_atom={locator.failed_atom_index} '
        f'attempted_binding={locator.attempted_binding}'
    )

## 3. Q5 Frontier Trace — where does the native where-body collapse?

`evaluate_native_where_frontier(...)` is an evaluator-layer probe. Given a where-body and the projected fact view, it returns the bindings that survive (`bindings`) plus the per-branch frontier where evaluation stopped (`frontier_rows`).

The phase below uses an unsatisfiable where (`age == 99`) and observes the frontier row at branch 0 atom 1 with failure kind `atom_filter_empty`.

In [ ]:
frontier_status = demo._phase_frontier(fixture, verbose=True)

print(f'\nfrontier_status = {frontier_status}')

## 4. Aggregate verification

`run_overlay_why_not_frontier_demo(...)` re-runs the three phases against a fresh fixture and returns the smoke-test status dict.

In [ ]:
summary = demo.run_overlay_why_not_frontier_demo(verbose=False)
expected = {
    'fact_overlay': 'passed',
    'why_not': 'completed',
    'frontier': 'atom_filter_empty',
}

print(f'Chapter summary : {summary}')
print(f'Expected        : {expected}')
assert summary == expected, summary
print('\n✓ Chapter 2 aggregate matches expected smoke contract.')

## Where to next

- **Chapter 3 (`03_proofframe_rule_overlays.ipynb`)** — Batch 4 ProofFrame Rechecker plus the three Batch 5 rule overlays (Disable / Literal Replace / Add Condition).
- **Module reference** — `src/kernel/application/docs/01_overview.md` for Fact Overlay / Why-not surface; `src/kernel/core/docs/01_architecture.md` for the evaluator-layer Frontier Trace.
- **Boundary** — Q3 / Q4 / Q5 are advanced-importable from `kernel.application` and `kernel.core.rules.ruleref_substrate`; v0.1 ships no SDK shells per the Batch 8 public-surface decision.